# Discrete DiGress: sampling with optional Energy Guidance

Run with **kernel = digress** conda env. Prefer **repo root** as cwd (where `configs/` exists), or set `REPO_ROOT` below.

- Set checkpoint path and optional Hydra overrides.
- Toggle `USE_GUIDANCE`, `lambda_scale`, `clip_grad_max`, and energy weights.
- Default Lightning / training paths do **not** use guidance unless you pass `guidance=`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch

# --- User: repository root (directory that contains configs/ and src/) ---
REPO_ROOT = Path("..").resolve()
if not (REPO_ROOT / "configs" / "config.yaml").is_file():
    REPO_ROOT = Path.cwd()

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

print("REPO_ROOT:", REPO_ROOT)

In [ ]:
# --- Checkpoint & Hydra ---
# Absolute path to .ckpt (same style as general.test_only in main.py)
CKPT_PATH = os.environ.get(
    "DIGRESS_CKPT",
    str(REPO_ROOT / "outputs" / "YOUR_RUN" / "checkpoints" / "last.ckpt"),
)

# Optional Hydra overrides (list of "key=value" strings), e.g. dataset paths
HYDRA_OVERRIDES: list[str] = []

from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

_cfg_dir = str(REPO_ROOT / "configs")
with initialize_config_dir(version_base="1.3", config_dir=_cfg_dir):
    cfg = compose(config_name="config", overrides=HYDRA_OVERRIDES)

assert cfg.model.type == "discrete", "This notebook targets DiscreteDenoisingDiffusion only."
OmegaConf.resolve(cfg)
print("dataset:", cfg.dataset.name)
print("ckpt:", CKPT_PATH)

In [ ]:
# --- Build dataset_infos + model kwargs (metabolite path; edit main.py branch if you use qm9 / etc.) ---
from metrics.molecular_metrics import SamplingMolecularMetrics
from metrics.molecular_metrics_discrete import TrainMolecularMetricsDiscrete
from diffusion.extra_features import DummyExtraFeatures, ExtraFeatures
from diffusion.extra_features_molecular import ExtraMolecularFeatures
from analysis.visualization import MolecularVisualization
from datasets.metabolite_dataset import MetaboliteDataModule, MetaboliteInfos, get_train_smiles
from diffusion_model_discrete import DiscreteDenoisingDiffusion

datamodule = MetaboliteDataModule(cfg)
dataset_infos = MetaboliteInfos(
    datamodule,
    cfg,
    recompute_statistics=cfg.general.get("recompute_statistics", False),
)
train_smiles = get_train_smiles(
    cfg,
    datamodule.train_dataloader(),
    dataset_infos,
    evaluate_dataset=cfg.general.get("evaluate_dataset", False),
)

if cfg.model.type == "discrete" and cfg.model.extra_features is not None:
    extra_features = ExtraFeatures(cfg.model.extra_features, dataset_info=dataset_infos)
    domain_features = ExtraMolecularFeatures(dataset_infos=dataset_infos)
else:
    extra_features = DummyExtraFeatures()
    domain_features = DummyExtraFeatures()

dataset_infos.compute_input_output_dims(
    datamodule=datamodule, extra_features=extra_features, domain_features=domain_features
)

train_metrics = TrainMolecularMetricsDiscrete(dataset_infos)
sampling_metrics = SamplingMolecularMetrics(dataset_infos, train_smiles)
visualization_tools = MolecularVisualization(cfg.dataset.remove_h, dataset_infos=dataset_infos)

model_kwargs = {
    "dataset_infos": dataset_infos,
    "train_metrics": train_metrics,
    "sampling_metrics": sampling_metrics,
    "visualization_tools": visualization_tools,
    "extra_features": extra_features,
    "domain_features": domain_features,
}

model = DiscreteDenoisingDiffusion.load_from_checkpoint(CKPT_PATH, **model_kwargs)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print("Loaded model on", device)

In [ ]:
# --- Guidance knobs (edit here) ---
USE_GUIDANCE = False
LAMBDA_SCALE = 1.0
CLIP_GRAD_MAX = 1.0

# One weight per energy term (must match number of BaseEnergy instances below).
# Example: single placeholder -> one weight.
ENERGY_WEIGHTS = [1.0]

from src.guidance import EnergyGuidance, CosineReferenceEnergy

dx = model.Xdim_output
reference_fp = torch.randn(dx, device=device)

energy_fns = [CosineReferenceEnergy(reference_fp.cpu())]  # buffers moved with .to(device)
guidance = EnergyGuidance(
    energy_fns,
    weights=ENERGY_WEIGHTS,
    lambda_scale=LAMBDA_SCALE,
    clip_grad_max=CLIP_GRAD_MAX,
).to(device)

g = guidance if USE_GUIDANCE else None
print("guidance:", "on" if g else "off")

In [ ]:
# --- Sample ---
BATCH_SIZE = 4
SAVE_FINAL = min(4, BATCH_SIZE)
KEEP_CHAIN = 0
NUMBER_CHAIN_STEPS = min(model.number_chain_steps, model.T - 1)

with torch.no_grad():
    samples = model.sample_batch(
        batch_id=0,
        batch_size=BATCH_SIZE,
        keep_chain=KEEP_CHAIN,
        number_chain_steps=NUMBER_CHAIN_STEPS,
        save_final=SAVE_FINAL,
        num_nodes=None,
        guidance=g,
    )

print("num molecules:", len(samples))
for i, (atoms, edges) in enumerate(samples[: min(3, len(samples))]):
    print(i, "N=", atoms.shape[0], "atom_shape", atoms.shape, "edge_shape", edges.shape)